# SFProxy Quick Run

This notebook matches the current `data/train/eval` config layout and the current repo paths on the 5090 machine.


In [ ]:
from pathlib import Path

REPO_ROOT = Path("/media/mengh/SharedData/zhanh/202604_midiproxy")
PROJECT_ROOT = REPO_ROOT / "synth-proxy"
WORKSPACE_BASE = Path("/media/mengh/SharedData/zhanh/202601_midisemi_data")
PIANO_SF2 = Path("/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/SalamanderGrandPiano-SF2/SalamanderGrandPiano-V3+20200602.sf2")
GUITAR_SF2 = Path("/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/SpanishClassicalGuitar-SF2/SpanishClassicalGuitar-20190618.sf2")

print(PROJECT_ROOT)
print(PIANO_SF2.exists(), GUITAR_SF2.exists())


## Export piano teacher data

This is the direct Python entrypoint version. For the single-run wrapper, use `run_scripts/train_sfproxy.sh`.


In [ ]:
!cd /media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy && \
python src/sfproxy/export_dataset_pkl.py \
  --config-name data_piano \
  paths.repo_root=/media/mengh/SharedData/zhanh/202604_midiproxy \
  paths.workspace_dir=/media/mengh/SharedData/zhanh/202601_midisemi_data \
  instrument.path=/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/SalamanderGrandPiano-SF2/SalamanderGrandPiano-V3+20200602.sf2 \
  dataset_size=2000 \
  start_index=0 end_index=2000 \
  seed_offset=1000


## Export guitar teacher data


In [ ]:
!cd /media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy && \
python src/sfproxy/export_dataset_pkl.py \
  --config-name data_guitar \
  paths.repo_root=/media/mengh/SharedData/zhanh/202604_midiproxy \
  paths.workspace_dir=/media/mengh/SharedData/zhanh/202601_midisemi_data \
  instrument.path=/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/SpanishClassicalGuitar-SF2/SpanishClassicalGuitar-20190618.sf2 \
  dataset_size=2000 \
  start_index=0 end_index=2000 \
  seed_offset=1000


## Build sampler statistics from external MIDI datasets


In [ ]:
# MAESTRO example
!cd /media/mengh/SharedData/zhanh/202601_midisemi/data_analysis && \
python dataset_midi_stats.py \
  --dataset MAESTRO_v3.0.0 \
  --root /path/to/maestro \
  --json-out-dir stats/midi_sampler \
  --output-dir figures/midi_sampler_stats \
  --no-show


In [ ]:
# GuitarSet example
!cd /media/mengh/SharedData/zhanh/202601_midisemi/data_analysis && \
python dataset_midi_stats.py \
  --dataset GuitarSet \
  --root /path/to/guitarset \
  --json-out-dir stats/midi_sampler \
  --output-dir figures/midi_sampler_stats \
  --no-show


## Train from exported folders

Replace the train/val export paths with the folders you just produced.


In [ ]:
import os
os.environ["WANDB_SILENT"] = "true"

!cd /media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy && \
python src/sfproxy/train.py \
  --config-name train \
  paths.repo_root=/media/mengh/SharedData/zhanh/202604_midiproxy \
  paths.workspace_dir=/media/mengh/SharedData/zhanh/202601_midisemi_data \
  dataset.train.path=/path/to/export_train_folder \
  dataset.val.path=/path/to/export_val_folder \
  dataset.name=piano \
  dataset.train.instrument_name=salamander_piano \
  dataset.val.instrument_name=salamander_piano


## Monotonic evaluation


In [ ]:
!cd /media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy && \
python src/sfproxy/eval.py \
  --config-name eval \
  mode=monotonic \
  paths.repo_root=/media/mengh/SharedData/zhanh/202604_midiproxy \
  paths.workspace_dir=/media/mengh/SharedData/zhanh/202601_midisemi_data \
  ckpt_path=/path/to/checkpoint.ckpt


## Velocity recovery evaluation


In [ ]:
!cd /media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy && \
python src/sfproxy/eval.py \
  --config-name eval \
  mode=velocity_recovery \
  paths.repo_root=/media/mengh/SharedData/zhanh/202604_midiproxy \
  paths.workspace_dir=/media/mengh/SharedData/zhanh/202601_midisemi_data \
  ckpt_path=/path/to/checkpoint.ckpt \
  device=cuda \
  velocity_recovery.instrument.path=/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/SalamanderGrandPiano-SF2/SalamanderGrandPiano-V3+20200602.sf2 \
  velocity_recovery.indomain.num_segments=50 \
  velocity_recovery.stress.num_segments=50


## Display the latest recovery plot


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

results = sorted((PROJECT_ROOT / "outputs").glob("**/velocity_recovery_results.json"))
if results:
    latest = results[-1]
    payload = json.loads(latest.read_text())
    ckpt_stem = Path(str(payload["ckpt_path"])).stem
    plot = latest.with_name(f"{ckpt_stem}_velocity_recovery_summary.png")
    print(latest)
    if plot.exists():
        display(Image(str(plot)))
    else:
        print(f"Plot not found: {plot}")
else:
    print("No velocity recovery outputs found under synth-proxy/outputs.")
